## GroundTruth 만든 것으로 평가
- gpt-4o-model(1차), 정성적 평가(2차)

### 1. 정답용 dataset 가져오기

In [1]:
import pandas as pd

df = pd.read_csv("data/groundTruth/final_groundTruth_f1.csv")
df.describe()

,user_info,recipe_info,generation,groundTruth
count,35,35,35,35
unique,10,35,35,35
top,"{'user_allergy_ingredients': [], 'user_dislike...","{'_id': '67610699846f9e5eb975e532', 'title': '...",{'main_changes_from_original_recipe': ['🥗 닭가슴살...,{'main_changes_from_original_recipe': ['🍗 연어 대...
freq,4,1,1,1


### 2. eval prompt로 test - toxicity
- 1,2,3번 모두 0.0

In [2]:
from langchain_teddynote import logging
# set_enable=False 로 지정하면 추적을 하지 않습니다.
logging.langsmith("랭체인 튜토리얼 프로젝트", set_enable=False)

LangSmith 추적을 하지 않습니다.


In [3]:
import nest_asyncio
import asyncio
import pandas as pd
import sys
sys.path.append('../')  # 상위 디렉토리의 src 폴더를 경로에 추가
from src.recipe_change_origin import eval_recipe

# nest_asyncio로 이미 실행 중인 루프에서 중첩 실행 허용
nest_asyncio.apply()

# 동시 요청 제한과 재시도 설정
MAX_CONCURRENT_REQUESTS = 3
RETRY_LIMIT = 3

async def generate(df, prompt_name, col_name):
    df[col_name] = None  # 결과 저장 열 생성

    # Semaphore 생성
    semaphore = asyncio.Semaphore(MAX_CONCURRENT_REQUESTS)

    tasks = []  # 모든 작업을 저장할 리스트

    for i in range(len(df)):
        retry_count = 0
        while retry_count <= RETRY_LIMIT:
            try:
                # Semaphore로 동시 요청 제한
                async with semaphore:
                    print(f"Processing row {i} (Attempt {retry_count + 1})...")
                    result = await eval_recipe(df.iloc[i].recipe_info, df.iloc[i].user_info, df.iloc[i].generation, df.iloc[i].groundTruth, prompt_name)
                    df.at[i, col_name] = result  # 결과 저장
                    break  # 성공하면 반복문 종료
            except Exception as e:
                retry_count += 1
                if retry_count > RETRY_LIMIT:
                    print(f"Failed to process row {i} after {RETRY_LIMIT} retries.")
                    break  # 재시도 초과 시 반복문 종료
                print(f"Error processing row {i}: {e}. Retrying...")
                await asyncio.sleep(10 ** retry_count)  # 재시도 전 대기

        tasks.append(asyncio.sleep(0))  # 리스트에 임시 작업 추가 (추후 확장 가능)

    await asyncio.gather(*tasks)  # 모든 작업 실행

    # 결과 확인
    print(df.head())
    


2024-12-23 05:24:12,730 - recipe_logger - INFO - LLM 초기화 완료: gpt-4o


In [3]:
# 데이터 로드
feature_num = 2  # 사용할 feature_num 설정
prompt_name = col_name = "toxicity"

file_path = f'data/groundTruth/final_groundTruth_f{feature_num}.csv'
df = pd.read_csv(file_path)

# 이벤트 루프 생성 및 실행
loop = asyncio.get_event_loop()
loop.run_until_complete(generate(df, prompt_name, col_name))

# 결과 저장
output_file = f'data/groundTruth/final_groundTruth_f{feature_num}.csv'
df.to_csv(output_file, index=False, encoding='utf-8-sig')
print(f"Processed data saved to {output_file}")

2024-12-23 05:13:07,887 - recipe_logger - INFO - json 출력 파서 초기화 완료.


Processing row 0 (Attempt 1)...


2024-12-23 05:13:08,207 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 05:13:08,211 - recipe_logger - INFO - LLM 레시피 생성 중...
2024-12-23 05:13:11,469 - recipe_logger - INFO - LLM 레시피 생성 완료
2024-12-23 05:13:11,481 - recipe_logger - INFO - json 출력 파서 초기화 완료.
2024-12-23 05:13:11,481 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 05:13:11,482 - recipe_logger - INFO - LLM 레시피 생성 중...


Processing row 1 (Attempt 1)...


2024-12-23 05:13:12,265 - recipe_logger - INFO - LLM 레시피 생성 완료
2024-12-23 05:13:12,277 - recipe_logger - INFO - json 출력 파서 초기화 완료.
2024-12-23 05:13:12,277 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 05:13:12,278 - recipe_logger - INFO - LLM 레시피 생성 중...


...

Processing row 33 (Attempt 1)...


2024-12-23 05:14:29,866 - recipe_logger - INFO - LLM 레시피 생성 완료
2024-12-23 05:14:29,882 - recipe_logger - INFO - json 출력 파서 초기화 완료.
2024-12-23 05:14:29,883 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 05:14:29,884 - recipe_logger - INFO - LLM 레시피 생성 중...


Processing row 34 (Attempt 1)...


2024-12-23 05:14:34,191 - recipe_logger - INFO - LLM 레시피 생성 완료


                                           user_info  \
0  {'user_allergy_ingredients': [], 'user_dislike...   
1  {'user_allergy_ingredients': [], 'user_dislike...   
2  {'user_allergy_ingredients': [], 'user_dislike...   
3  {'user_allergy_ingredients': [], 'user_dislike...   
4  {'user_allergy_ingredients': ['우유', '견과류'], 'u...   

                                         recipe_info  \
0  {'_id': '67610699846f9e5eb975e532', 'title': '...   
1  {'_id': '6761069a846f9e5eb9761021', 'title': '...   
2  {'_id': '6761069a846f9e5eb9760836', 'title': '...   
3  {'_id': '6761069a846f9e5eb9760639', 'title': '...   
4  {'_id': '6761069a846f9e5eb976093b', 'title': '...   

                                          generation  \
0  {'main_changes_from_original_recipe': ['연어 대신 ...   
1  {'main_changes_from_original_recipe': ['🍇 포도주스...   
2  {'main_changes_from_original_recipe': ['양파와 감자...   
3  {'main_changes_from_original_recipe': ['레몬을 굵은...   
4  {'main_changes_from_original_recipe': ['소고기

In [8]:
import pandas as pd
import ast

col_name = "toxicity"

# 문자열이 아닌 값도 처리하기 위한 함수 정의
def safe_literal_eval(val):
    if isinstance(val, str):
        return ast.literal_eval(val)  # 문자열이면 딕셔너리로 변환
    return val  # 이미 딕셔너리면 그대로 반환

df[col_name] = df[col_name].apply(safe_literal_eval)

# 1. score 평균값 계산
df['score'] = df[col_name].apply(lambda x: x['score'])
score_mean = df['score'].mean()

# 2. reason이 '해당없음'가 아닌 행
df['reason'] = df[col_name].apply(lambda x: x['reason'])
non_x_reasons = df[df['reason'] != '해당없음']

# 3. score 상위 3개의 행
top3_scores = df.nlargest(3, 'score')

# 결과 출력
print(f"Score 평균값: {score_mean}")
print("\nReason이 '해당없음'가 아닌 행:")
print(non_x_reasons)
print("\nScore 상위 3개의 행:")
print(top3_scores)


Score 평균값: 0.005714285714285714

Reason이 '해당없음'가 아닌 행:
Empty DataFrame
Columns: [user_info, recipe_info, generation, groundTruth, toxicity, score, reason]
Index: []

Score 상위 3개의 행:
                                           user_info  \
3  {'user_allergy_ingredients': [], 'user_dislike...   
0  {'user_allergy_ingredients': [], 'user_dislike...   
1  {'user_allergy_ingredients': [], 'user_dislike...   

                                         recipe_info  \
3  {'_id': '6761069a846f9e5eb9760639', 'title': '...   
0  {'_id': '67610699846f9e5eb975e532', 'title': '...   
1  {'_id': '6761069a846f9e5eb9761021', 'title': '...   

                                          generation  \
3  {'original_recipe_food_group_composition': [{'...   
0  {'original_recipe_food_group_composition': [{'...   
1  {'original_recipe_food_group_composition': [{'...   

                                         groundTruth  \
3  {'original_recipe_food_group_composition': [{'...   
0  {'original_recipe_food_group

In [8]:
# 데이터 로드
feature_num = 3  # 사용할 feature_num 설정
prompt_name = col_name = "toxicity"

file_path = f'data/groundTruth/final_groundTruth_f{feature_num}.csv'
df = pd.read_csv(file_path)

# 이벤트 루프 생성 및 실행
loop = asyncio.get_event_loop()
loop.run_until_complete(generate(df, prompt_name, col_name))

# 결과 저장
output_file = f'data/groundTruth/final_groundTruth_f{feature_num}.csv'
df.to_csv(output_file, index=False, encoding='utf-8-sig')
print(f"Processed data saved to {output_file}")

2024-12-23 05:25:51,264 - recipe_logger - INFO - json 출력 파서 초기화 완료.
2024-12-23 05:25:51,264 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 05:25:51,266 - recipe_logger - INFO - LLM 레시피 생성 중...


Processing row 0 (Attempt 1)...


2024-12-23 05:25:52,135 - recipe_logger - INFO - LLM 레시피 생성 완료
2024-12-23 05:25:52,149 - recipe_logger - INFO - json 출력 파서 초기화 완료.
2024-12-23 05:25:52,150 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 05:25:52,151 - recipe_logger - INFO - LLM 레시피 생성 중...


Processing row 1 (Attempt 1)...


2024-12-23 05:25:53,021 - recipe_logger - INFO - LLM 레시피 생성 완료
2024-12-23 05:25:53,031 - recipe_logger - INFO - json 출력 파서 초기화 완료.
2024-12-23 05:25:53,032 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 05:25:53,033 - recipe_logger - INFO - LLM 레시피 생성 중...


...

Processing row 33 (Attempt 1)...


2024-12-23 05:27:18,085 - recipe_logger - INFO - LLM 레시피 생성 완료
2024-12-23 05:27:18,101 - recipe_logger - INFO - json 출력 파서 초기화 완료.
2024-12-23 05:27:18,102 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 05:27:18,103 - recipe_logger - INFO - LLM 레시피 생성 중...


Processing row 34 (Attempt 1)...


2024-12-23 05:27:22,276 - recipe_logger - INFO - LLM 레시피 생성 완료


                                           user_info  \
0  {'user_allergy_ingredients': [], 'user_dislike...   
1  {'user_allergy_ingredients': [], 'user_dislike...   
2  {'user_allergy_ingredients': [], 'user_dislike...   
3  {'user_allergy_ingredients': [], 'user_dislike...   
4  {'user_allergy_ingredients': ['우유', '견과류'], 'u...   

                                         recipe_info  \
0  {'_id': '67610699846f9e5eb975e532', 'title': '...   
1  {'_id': '6761069a846f9e5eb9761021', 'title': '...   
2  {'_id': '6761069a846f9e5eb9760836', 'title': '...   
3  {'_id': '6761069a846f9e5eb9760639', 'title': '...   
4  {'_id': '6761069a846f9e5eb976093b', 'title': '...   

                                          generation  \
0  {'original_recipe_food_group_composition': [{'...   
1  {'original_recipe_food_group_composition': [{'...   
2  {'original_recipe_food_group_composition': [{'...   
3  {'original_recipe_food_group_composition': [{'...   
4  {'original_recipe_food_group_composition': 

In [9]:
import pandas as pd
import ast

# 데이터 로드
feature_num = 3  # 사용할 feature_num 설정
prompt_name = col_name = "toxicity"

file_path = f'data/groundTruth/final_groundTruth_f{feature_num}.csv'
df = pd.read_csv(file_path)

# 문자열이 아닌 값도 처리하기 위한 함수 정의
def safe_literal_eval(val):
    if isinstance(val, str):
        return ast.literal_eval(val)  # 문자열이면 딕셔너리로 변환
    return val  # 이미 딕셔너리면 그대로 반환

df[col_name] = df[col_name].apply(safe_literal_eval)

# 1. score 평균값 계산
df['score'] = df[col_name].apply(lambda x: x['score'])
score_mean = df['score'].mean()

# 2. reason이 '해당없음'가 아닌 행
df['reason'] = df[col_name].apply(lambda x: x['reason'])
non_x_reasons = df[df['reason'] != '해당없음']

# 3. score 상위 3개의 행
top3_scores = df.nlargest(3, 'score')

# 결과 출력
print(f"Score 평균값: {score_mean}")
print("\nReason이 '해당없음'가 아닌 행:")
print(non_x_reasons)
print("\nScore 상위 3개의 행:")
print(top3_scores)


Score 평균값: 0.0

Reason이 '해당없음'가 아닌 행:
Empty DataFrame
Columns: [user_info, recipe_info, generation, groundTruth, toxicity, score, reason]
Index: []

Score 상위 3개의 행:
                                           user_info  \
0  {'user_allergy_ingredients': [], 'user_dislike...   
1  {'user_allergy_ingredients': [], 'user_dislike...   
2  {'user_allergy_ingredients': [], 'user_dislike...   

                                         recipe_info  \
0  {'_id': '67610699846f9e5eb975e532', 'title': '...   
1  {'_id': '6761069a846f9e5eb9761021', 'title': '...   
2  {'_id': '6761069a846f9e5eb9760836', 'title': '...   

                                          generation  \
0  {'original_recipe_food_group_composition': [{'...   
1  {'original_recipe_food_group_composition': [{'...   
2  {'original_recipe_food_group_composition': [{'...   

                                         groundTruth  \
0  {'original_recipe_food_group_composition': [{'...   
1  {'original_recipe_food_group_composition': [{

### 2. eval prompt로 test - contextcorrectness
- 1,2,3번 모두 0.0

In [1]:
from langchain_teddynote import logging
# set_enable=False 로 지정하면 추적을 하지 않습니다.
logging.langsmith("랭체인 튜토리얼 프로젝트", set_enable=False)

import nest_asyncio
import asyncio
import pandas as pd
import sys
sys.path.append('../')  # 상위 디렉토리의 src 폴더를 경로에 추가
from src.recipe_change_origin import eval_recipe

# nest_asyncio로 이미 실행 중인 루프에서 중첩 실행 허용
nest_asyncio.apply()

# 동시 요청 제한과 재시도 설정
MAX_CONCURRENT_REQUESTS = 3
RETRY_LIMIT = 3

async def generate(df, prompt_name, col_name):
    df[col_name] = None  # 결과 저장 열 생성

    # Semaphore 생성
    semaphore = asyncio.Semaphore(MAX_CONCURRENT_REQUESTS)

    tasks = []  # 모든 작업을 저장할 리스트

    for i in range(len(df)):
        retry_count = 0
        while retry_count <= RETRY_LIMIT:
            try:
                # Semaphore로 동시 요청 제한
                async with semaphore:
                    print(f"Processing row {i} (Attempt {retry_count + 1})...")
                    result = await eval_recipe(df.iloc[i].recipe_info, df.iloc[i].user_info, df.iloc[i].generation, df.iloc[i].groundTruth, prompt_name)
                    df.at[i, col_name] = result  # 결과 저장
                    break  # 성공하면 반복문 종료
            except Exception as e:
                retry_count += 1
                if retry_count > RETRY_LIMIT:
                    print(f"Failed to process row {i} after {RETRY_LIMIT} retries.")
                    break  # 재시도 초과 시 반복문 종료
                print(f"Error processing row {i}: {e}. Retrying...")
                await asyncio.sleep(10 ** retry_count)  # 재시도 전 대기

        tasks.append(asyncio.sleep(0))  # 리스트에 임시 작업 추가 (추후 확장 가능)

    await asyncio.gather(*tasks)  # 모든 작업 실행

    # 결과 확인
    print(df.head())

LangSmith 추적을 하지 않습니다.


2024-12-23 06:17:40,609 - recipe_logger - INFO - LLM 초기화 완료: gpt-4o


In [3]:
# 데이터 로드
feature_num = 1  # 사용할 feature_num 설정
prompt_name = col_name = "contextcorrectness"

file_path = f'data/groundTruth/final_groundTruth_f{feature_num}.csv'
df = pd.read_csv(file_path)

# 이벤트 루프 생성 및 실행
loop = asyncio.get_event_loop()
loop.run_until_complete(generate(df, prompt_name, col_name))

# 결과 저장
output_file = f'data/groundTruth/final_groundTruth_f{feature_num}.csv'
df.to_csv(output_file, index=False, encoding='utf-8-sig')
print(f"Processed data saved to {output_file}")

2024-12-23 06:17:53,234 - recipe_logger - INFO - json 출력 파서 초기화 완료.
2024-12-23 06:17:53,235 - recipe_logger - INFO - langfuse prompt template 생성 완료


2024-12-23 06:17:53,235 - recipe_logger - INFO - LLM 레시피 생성 중...


Processing row 0 (Attempt 1)...


2024-12-23 06:17:54,692 - recipe_logger - INFO - LLM 레시피 생성 완료
2024-12-23 06:17:54,704 - recipe_logger - INFO - json 출력 파서 초기화 완료.
2024-12-23 06:17:54,705 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 06:17:54,706 - recipe_logger - INFO - LLM 레시피 생성 중...


Processing row 1 (Attempt 1)...


...

Processing row 33 (Attempt 1)...


2024-12-23 06:18:44,795 - recipe_logger - INFO - LLM 레시피 생성 완료
2024-12-23 06:18:44,809 - recipe_logger - INFO - json 출력 파서 초기화 완료.
2024-12-23 06:18:44,809 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 06:18:44,810 - recipe_logger - INFO - LLM 레시피 생성 중...


Processing row 34 (Attempt 1)...


2024-12-23 06:18:46,404 - recipe_logger - INFO - LLM 레시피 생성 완료


                                           user_info  \
0  {'user_allergy_ingredients': [], 'user_dislike...   
1  {'user_allergy_ingredients': [], 'user_dislike...   
2  {'user_allergy_ingredients': [], 'user_dislike...   
3  {'user_allergy_ingredients': [], 'user_dislike...   
4  {'user_allergy_ingredients': ['우유', '견과류'], 'u...   

                                         recipe_info  \
0  {'_id': '67610699846f9e5eb975e532', 'title': '...   
1  {'_id': '6761069a846f9e5eb9761021', 'title': '...   
2  {'_id': '6761069a846f9e5eb9760836', 'title': '...   
3  {'_id': '6761069a846f9e5eb9760639', 'title': '...   
4  {'_id': '6761069a846f9e5eb976093b', 'title': '...   

                                          generation  \
0  {'main_changes_from_original_recipe': ['🥗 닭가슴살...   
1  {'main_changes_from_original_recipe': ['🍇 포도주스...   
2  {'main_changes_from_original_recipe': ['🥣 양파는 ...   
3  {'main_changes_from_original_recipe': ['🍋 레몬을 ...   
4  {'main_changes_from_original_recipe': ['🥣 황

In [4]:
import pandas as pd
import ast

# 데이터 로드
feature_num = 1  # 사용할 feature_num 설정
prompt_name = col_name = "contextcorrectness"

file_path = f'data/groundTruth/final_groundTruth_f{feature_num}.csv'
df = pd.read_csv(file_path)

# 문자열이 아닌 값도 처리하기 위한 함수 정의
def safe_literal_eval(val):
    if isinstance(val, str):
        return ast.literal_eval(val)  # 문자열이면 딕셔너리로 변환
    return val  # 이미 딕셔너리면 그대로 반환

df[col_name] = df[col_name].apply(safe_literal_eval)

# 1. score 평균값 계산
df[col_name+'score'] = df[col_name].apply(lambda x: x['score'])
score_mean = df[col_name+'score'].mean()

# 3. score 하위 10개의 행
top3_scores = df.nsmallest(10, col_name+'score')

# 결과 출력
print(f"Score 평균값: {score_mean}")
print("\nScore 하위 10개의 행:")
print(top3_scores)

Score 평균값: 0.8699999999999999

Score 하위 10개의 행:
                                            user_info  \
17  {'user_allergy_ingredients': ['우유', '크림', '버터'...   
27  {'user_allergy_ingredients': ['우유', '크림', '버터'...   
33  {'user_allergy_ingredients': [], 'user_dislike...   
4   {'user_allergy_ingredients': ['우유', '견과류'], 'u...   
0   {'user_allergy_ingredients': [], 'user_dislike...   
1   {'user_allergy_ingredients': [], 'user_dislike...   
2   {'user_allergy_ingredients': [], 'user_dislike...   
3   {'user_allergy_ingredients': [], 'user_dislike...   
6   {'user_allergy_ingredients': [], 'user_dislike...   
7   {'user_allergy_ingredients': ['우유', '크림', '버터'...   

                                          recipe_info  \
17  {'_id': '6761069a846f9e5eb97606ed', 'title': '...   
27  {'_id': '6761069a846f9e5eb97610a5', 'title': '...   
33  {'_id': '6761069a846f9e5eb9761f26', 'title': '...   
4   {'_id': '6761069a846f9e5eb976093b', 'title': '...   
0   {'_id': '67610699846f9e5eb975e532',

In [13]:
df.iloc[10]

user_info                  {'user_allergy_ingredients': [], 'user_dislike...
recipe_info                {'_id': '6761069a846f9e5eb9761fe5', 'title': '...
generation                 {'main_changes_from_original_recipe': ['🥣 닭가슴살...
groundTruth                {'main_changes_from_original_recipe': ['🥦 브로콜리...
contextcorrectness         {'score': 0.2, 'reason': 'The generation does ...
contextcorrectnessscore                                                  0.2
Name: 10, dtype: object

In [5]:
# 데이터 로드
feature_num = 2  # 사용할 feature_num 설정
prompt_name = col_name = "contextcorrectness"

file_path = f'data/groundTruth/final_groundTruth_f{feature_num}.csv'
df = pd.read_csv(file_path)

# 이벤트 루프 생성 및 실행
loop = asyncio.get_event_loop()
loop.run_until_complete(generate(df, prompt_name, col_name))

# 결과 저장
output_file = f'data/groundTruth/final_groundTruth_f{feature_num}.csv'
df.to_csv(output_file, index=False, encoding='utf-8-sig')
print(f"Processed data saved to {output_file}")

2024-12-23 06:19:51,606 - recipe_logger - INFO - json 출력 파서 초기화 완료.
2024-12-23 06:19:51,607 - recipe_logger - INFO - langfuse prompt template 생성 완료


2024-12-23 06:19:51,615 - recipe_logger - INFO - LLM 레시피 생성 중...


Processing row 0 (Attempt 1)...


2024-12-23 06:19:52,967 - recipe_logger - INFO - LLM 레시피 생성 완료
2024-12-23 06:19:52,978 - recipe_logger - INFO - json 출력 파서 초기화 완료.
2024-12-23 06:19:52,979 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 06:19:52,980 - recipe_logger - INFO - LLM 레시피 생성 중...


Processing row 1 (Attempt 1)...


...

Processing row 33 (Attempt 1)...


2024-12-23 06:20:42,425 - recipe_logger - INFO - LLM 레시피 생성 완료
2024-12-23 06:20:42,437 - recipe_logger - INFO - json 출력 파서 초기화 완료.
2024-12-23 06:20:42,438 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 06:20:42,439 - recipe_logger - INFO - LLM 레시피 생성 중...


Processing row 34 (Attempt 1)...


2024-12-23 06:20:43,765 - recipe_logger - INFO - LLM 레시피 생성 완료


                                           user_info  \
0  {'user_allergy_ingredients': [], 'user_dislike...   
1  {'user_allergy_ingredients': [], 'user_dislike...   
2  {'user_allergy_ingredients': [], 'user_dislike...   
3  {'user_allergy_ingredients': [], 'user_dislike...   
4  {'user_allergy_ingredients': ['우유', '견과류'], 'u...   

                                         recipe_info  \
0  {'_id': '67610699846f9e5eb975e532', 'title': '...   
1  {'_id': '6761069a846f9e5eb9761021', 'title': '...   
2  {'_id': '6761069a846f9e5eb9760836', 'title': '...   
3  {'_id': '6761069a846f9e5eb9760639', 'title': '...   
4  {'_id': '6761069a846f9e5eb976093b', 'title': '...   

                                          generation  \
0  {'main_changes_from_original_recipe': ['연어 대신 ...   
1  {'main_changes_from_original_recipe': ['🍇 포도주스...   
2  {'main_changes_from_original_recipe': ['양파와 감자...   
3  {'main_changes_from_original_recipe': ['레몬을 굵은...   
4  {'main_changes_from_original_recipe': ['소고기

In [6]:
import pandas as pd
import ast

# 데이터 로드
feature_num = 2  # 사용할 feature_num 설정
prompt_name = col_name = "contextcorrectness"

file_path = f'data/groundTruth/final_groundTruth_f{feature_num}.csv'
df = pd.read_csv(file_path)

# 문자열이 아닌 값도 처리하기 위한 함수 정의
def safe_literal_eval(val):
    if isinstance(val, str):
        return ast.literal_eval(val)  # 문자열이면 딕셔너리로 변환
    return val  # 이미 딕셔너리면 그대로 반환

df[col_name] = df[col_name].apply(safe_literal_eval)

# 1. score 평균값 계산
df[col_name+'score'] = df[col_name].apply(lambda x: x['score'])
score_mean = df[col_name+'score'].mean()

# 3. score 하위 10개의 행
top3_scores = df.nsmallest(10, col_name+'score')

# 결과 출력
print(f"Score 평균값: {score_mean}")
print("\nScore 하위 10개의 행:")
print(top3_scores)

Score 평균값: 0.8885714285714285

Score 하위 10개의 행:
                                            user_info  \
6   {'user_allergy_ingredients': [], 'user_dislike...   
8   {'user_allergy_ingredients': [], 'user_dislike...   
13  {'user_allergy_ingredients': [], 'user_dislike...   
17  {'user_allergy_ingredients': ['우유', '크림', '버터'...   
27  {'user_allergy_ingredients': ['우유', '크림', '버터'...   
29  {'user_allergy_ingredients': ['새우', '게', '견과류'...   
0   {'user_allergy_ingredients': [], 'user_dislike...   
1   {'user_allergy_ingredients': [], 'user_dislike...   
2   {'user_allergy_ingredients': [], 'user_dislike...   
3   {'user_allergy_ingredients': [], 'user_dislike...   

                                          recipe_info  \
6   {'_id': '6761069a846f9e5eb975f670', 'title': '...   
8   {'_id': '6761069a846f9e5eb9760f44', 'title': '...   
13  {'_id': '6761069a846f9e5eb97622b1', 'title': '...   
17  {'_id': '6761069a846f9e5eb97606ed', 'title': '...   
27  {'_id': '6761069a846f9e5eb97610a5',

In [7]:
# 데이터 로드
feature_num = 3  # 사용할 feature_num 설정
prompt_name = col_name = "contextcorrectness"

file_path = f'data/groundTruth/final_groundTruth_f{feature_num}.csv'
df = pd.read_csv(file_path)

# 이벤트 루프 생성 및 실행
loop = asyncio.get_event_loop()
loop.run_until_complete(generate(df, prompt_name, col_name))

# 결과 저장
output_file = f'data/groundTruth/final_groundTruth_f{feature_num}.csv'
df.to_csv(output_file, index=False, encoding='utf-8-sig')
print(f"Processed data saved to {output_file}")

2024-12-23 06:20:43,973 - recipe_logger - INFO - json 출력 파서 초기화 완료.


Processing row 0 (Attempt 1)...


2024-12-23 06:20:43,975 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 06:20:43,982 - recipe_logger - INFO - LLM 레시피 생성 중...
2024-12-23 06:20:45,498 - recipe_logger - INFO - LLM 레시피 생성 완료
2024-12-23 06:20:45,512 - recipe_logger - INFO - json 출력 파서 초기화 완료.
2024-12-23 06:20:45,513 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 06:20:45,514 - recipe_logger - INFO - LLM 레시피 생성 중...


Processing row 1 (Attempt 1)...


2024-12-23 06:20:47,135 - recipe_logger - INFO - LLM 레시피 생성 완료
2024-12-23 06:20:47,148 - recipe_logger - INFO - json 출력 파서 초기화 완료.
2024-12-23 06:20:47,148 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 06:20:47,149 - recipe_logger - INFO - LLM 레시피 생성 중...


...

Processing row 33 (Attempt 1)...


2024-12-23 06:21:58,000 - recipe_logger - INFO - LLM 레시피 생성 완료
2024-12-23 06:21:58,018 - recipe_logger - INFO - json 출력 파서 초기화 완료.
2024-12-23 06:21:58,018 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 06:21:58,020 - recipe_logger - INFO - LLM 레시피 생성 중...


Processing row 34 (Attempt 1)...


2024-12-23 06:22:00,972 - recipe_logger - INFO - LLM 레시피 생성 완료


                                           user_info  \
0  {'user_allergy_ingredients': [], 'user_dislike...   
1  {'user_allergy_ingredients': [], 'user_dislike...   
2  {'user_allergy_ingredients': [], 'user_dislike...   
3  {'user_allergy_ingredients': [], 'user_dislike...   
4  {'user_allergy_ingredients': ['우유', '견과류'], 'u...   

                                         recipe_info  \
0  {'_id': '67610699846f9e5eb975e532', 'title': '...   
1  {'_id': '6761069a846f9e5eb9761021', 'title': '...   
2  {'_id': '6761069a846f9e5eb9760836', 'title': '...   
3  {'_id': '6761069a846f9e5eb9760639', 'title': '...   
4  {'_id': '6761069a846f9e5eb976093b', 'title': '...   

                                          generation  \
0  {'original_recipe_food_group_composition': [{'...   
1  {'original_recipe_food_group_composition': [{'...   
2  {'original_recipe_food_group_composition': [{'...   
3  {'original_recipe_food_group_composition': [{'...   
4  {'original_recipe_food_group_composition': 

In [9]:
import pandas as pd
import ast

# 데이터 로드
feature_num = 3  # 사용할 feature_num 설정
prompt_name = col_name = "contextcorrectness"

file_path = f'data/groundTruth/final_groundTruth_f{feature_num}.csv'
df = pd.read_csv(file_path)

# 문자열이 아닌 값도 처리하기 위한 함수 정의
def safe_literal_eval(val):
    if isinstance(val, str):
        return ast.literal_eval(val)  # 문자열이면 딕셔너리로 변환
    return val  # 이미 딕셔너리면 그대로 반환

df[col_name] = df[col_name].apply(safe_literal_eval)

# 1. score 평균값 계산
df[col_name+'score'] = df[col_name].apply(lambda x: x['score'])
score_mean = df[col_name+'score'].mean()

# 3. score 하위 10개의 행
top3_scores = df.nsmallest(10, col_name+'score')

# 결과 출력
print(f"Score 평균값: {score_mean}")
print("\nScore 하위 10개의 행:")
print(top3_scores)

Score 평균값: 0.8542857142857143

Score 하위 10개의 행:
                                            user_info  \
5   {'user_allergy_ingredients': [], 'user_dislike...   
17  {'user_allergy_ingredients': ['우유', '크림', '버터'...   
29  {'user_allergy_ingredients': ['새우', '게', '견과류'...   
2   {'user_allergy_ingredients': [], 'user_dislike...   
3   {'user_allergy_ingredients': [], 'user_dislike...   
8   {'user_allergy_ingredients': [], 'user_dislike...   
9   {'user_allergy_ingredients': ['새우', '게', '견과류'...   
11  {'user_allergy_ingredients': [], 'user_dislike...   
34  {'user_allergy_ingredients': ['우유', '견과류'], 'u...   
1   {'user_allergy_ingredients': [], 'user_dislike...   

                                          recipe_info  \
5   {'_id': '67610699846f9e5eb975e745', 'title': '...   
17  {'_id': '6761069a846f9e5eb97606ed', 'title': '...   
29  {'_id': '6761069a846f9e5eb97600e1', 'title': '...   
2   {'_id': '6761069a846f9e5eb9760836', 'title': '...   
3   {'_id': '6761069a846f9e5eb9760639',

### 2. eval prompt로 test - relevance
- 1,2,3번 모두 0.0

In [1]:
from langchain_teddynote import logging
# set_enable=False 로 지정하면 추적을 하지 않습니다.
logging.langsmith("랭체인 튜토리얼 프로젝트", set_enable=False)

import nest_asyncio
import asyncio
import pandas as pd
import sys
sys.path.append('../')  # 상위 디렉토리의 src 폴더를 경로에 추가
from src.recipe_change_origin import eval_recipe

# nest_asyncio로 이미 실행 중인 루프에서 중첩 실행 허용
nest_asyncio.apply()

# 동시 요청 제한과 재시도 설정
MAX_CONCURRENT_REQUESTS = 3
RETRY_LIMIT = 3

async def generate(df, prompt_name, col_name):
    df[col_name] = None  # 결과 저장 열 생성

    # Semaphore 생성
    semaphore = asyncio.Semaphore(MAX_CONCURRENT_REQUESTS)

    tasks = []  # 모든 작업을 저장할 리스트

    for i in range(len(df)):
        retry_count = 0
        while retry_count <= RETRY_LIMIT:
            try:
                # Semaphore로 동시 요청 제한
                async with semaphore:
                    print(f"Processing row {i} (Attempt {retry_count + 1})...")
                    result = await eval_recipe(df.iloc[i].recipe_info, df.iloc[i].user_info, df.iloc[i].generation, df.iloc[i].groundTruth, prompt_name)
                    df.at[i, col_name] = result  # 결과 저장
                    break  # 성공하면 반복문 종료
            except Exception as e:
                retry_count += 1
                if retry_count > RETRY_LIMIT:
                    print(f"Failed to process row {i} after {RETRY_LIMIT} retries.")
                    break  # 재시도 초과 시 반복문 종료
                print(f"Error processing row {i}: {e}. Retrying...")
                await asyncio.sleep(10 ** retry_count)  # 재시도 전 대기

        tasks.append(asyncio.sleep(0))  # 리스트에 임시 작업 추가 (추후 확장 가능)

    await asyncio.gather(*tasks)  # 모든 작업 실행

    # 결과 확인
    print(df.head())
    


LangSmith 추적을 하지 않습니다.


2024-12-23 07:42:37,806 - recipe_logger - INFO - LLM 초기화 완료: gpt-4o


In [2]:
# 데이터 로드
feature_num = 1  # 사용할 feature_num 설정
prompt_name = col_name = "relevance"

file_path = f'data/groundTruth/final_groundTruth_f{feature_num}.csv'
df = pd.read_csv(file_path)

# 이벤트 루프 생성 및 실행
loop = asyncio.get_event_loop()
loop.run_until_complete(generate(df, prompt_name, col_name))

# 결과 저장
output_file = f'data/groundTruth/final_groundTruth_f{feature_num}.csv'
df.to_csv(output_file, index=False, encoding='utf-8-sig')
print(f"Processed data saved to {output_file}")

2024-12-23 07:42:41,147 - recipe_logger - INFO - json 출력 파서 초기화 완료.


Processing row 0 (Attempt 1)...


2024-12-23 07:42:41,725 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 07:42:41,729 - recipe_logger - INFO - LLM 레시피 생성 중...
2024-12-23 07:42:45,639 - recipe_logger - INFO - LLM 레시피 생성 완료
2024-12-23 07:42:45,649 - recipe_logger - INFO - json 출력 파서 초기화 완료.
2024-12-23 07:42:45,650 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 07:42:45,651 - recipe_logger - INFO - LLM 레시피 생성 중...


Processing row 1 (Attempt 1)...


2024-12-23 07:42:47,027 - recipe_logger - INFO - LLM 레시피 생성 완료
2024-12-23 07:42:47,042 - recipe_logger - INFO - json 출력 파서 초기화 완료.
2024-12-23 07:42:47,043 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 07:42:47,044 - recipe_logger - INFO - LLM 레시피 생성 중...


...

Processing row 33 (Attempt 1)...


2024-12-23 07:45:10,605 - recipe_logger - INFO - LLM 레시피 생성 완료
2024-12-23 07:45:10,622 - recipe_logger - INFO - json 출력 파서 초기화 완료.
2024-12-23 07:45:10,623 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 07:45:10,624 - recipe_logger - INFO - LLM 레시피 생성 중...


Processing row 34 (Attempt 1)...


2024-12-23 07:45:19,770 - recipe_logger - INFO - LLM 레시피 생성 완료


                                           user_info  \
0  {'user_allergy_ingredients': [], 'user_dislike...   
1  {'user_allergy_ingredients': [], 'user_dislike...   
2  {'user_allergy_ingredients': [], 'user_dislike...   
3  {'user_allergy_ingredients': [], 'user_dislike...   
4  {'user_allergy_ingredients': ['우유', '견과류'], 'u...   

                                         recipe_info  \
0  {'_id': '67610699846f9e5eb975e532', 'title': '...   
1  {'_id': '6761069a846f9e5eb9761021', 'title': '...   
2  {'_id': '6761069a846f9e5eb9760836', 'title': '...   
3  {'_id': '6761069a846f9e5eb9760639', 'title': '...   
4  {'_id': '6761069a846f9e5eb976093b', 'title': '...   

                                          generation  \
0  {'main_changes_from_original_recipe': ['🥗 닭가슴살...   
1  {'main_changes_from_original_recipe': ['🍇 포도주스...   
2  {'main_changes_from_original_recipe': ['🥣 양파는 ...   
3  {'main_changes_from_original_recipe': ['🍋 레몬을 ...   
4  {'main_changes_from_original_recipe': ['🥣 황

In [5]:
import pandas as pd
import ast

# 데이터 로드
feature_num = 1  # 사용할 feature_num 설정
prompt_name = col_name = "relevance"

file_path = f'data/groundTruth/final_groundTruth_f{feature_num}.csv'
df = pd.read_csv(file_path)

# 문자열이 아닌 값도 처리하기 위한 함수 정의
def safe_literal_eval(val):
    if isinstance(val, str):
        return ast.literal_eval(val)  # 문자열이면 딕셔너리로 변환
    return val  # 이미 딕셔너리면 그대로 반환

df[col_name] = df[col_name].apply(safe_literal_eval)

# 1. score 평균값 계산
df['score'] = df[col_name].apply(lambda x: x['score'])
score_mean = df['score'].mean()

# 2. reason이 '해당없음'가 아닌 행
df['reason'] = df[col_name].apply(lambda x: x['reason'])
non_x_reasons = df[df['reason'] != '해당없음']

# 3. score 상위 3개의 행
top3_scores = df.nsmallest(10, 'score')

# 결과 출력
print(f"Score 평균값: {score_mean}")

for i in range(len(top3_scores)):
    print(top3_scores.iloc[i].user_info)
    print(top3_scores.iloc[i].recipe_info)
    print(top3_scores.iloc[i].generation)
    print(top3_scores.iloc[i].score)
    print(top3_scores.iloc[i].reason)
    print()


Score 평균값: 0.8057142857142858
{'user_allergy_ingredients': ['새우', '게', '견과류'], 'user_dislike_ingredients': ['닭고기', '브로콜리', '오이'], 'user_spicy_level': '5단계', 'user_cooking_level': '고급', 'user_owned_ingredients': ['연어', '아스파라거스', '파스타면'], 'user_basic_seasoning': ['버터', '허브솔트', '파마산 치즈'], 'must_use_ingredients': ['아스파라거스']}
{'_id': '6761069a846f9e5eb976030a', 'title': '\ufeff편스토랑이찬원 멸치고추다짐장 레시피 만드는법', 'type_key': '반찬', 'method_key': '조림', 'servings': '4인분', 'cooking_time': '30분 이내', 'difficulty': '초급', 'ingredients': ['멸치(60마리)', '아삭이고추(2개)', '홍고추(2개)', '청양고추(16개)', '식용유(4큰술)', '다진 마늘(4큰술)', '국간장(4큰술)', '멸치액젓(3큰술)', '매실액(1큰술)', '참기름(2큰술)', '물(1/2컵)'], 'cooking_steps': ['멸치를 내장과 머리를 제거하고 다져 줍니다. 그리고 마른 팬에 5분 정도 볶아서 비린내와 수분을 날려 주세요.약 불에서 볶아 주세요.', '고추는 모두 잘게 다져 줍니다.', '다시마로 육수를 내어 줍니다.센 불에서 끓여 주세요.', '달궈 진 팬에 식용유를 두르고 다진 마늘을 넣어서 볶아 줍니다. 그리고 식용유에 마늘 향이 베이면 다진 고추를 넣고 볶아 주세요.중 약불에서 볶아 주세요', '고추가 어느 정도 볶아지면 멸치를 넣은 후 양념 재료와 다시마 육수를 적당히 넣어서 볶아 줍니다.중 약불에서 볶아 주세요.', ' 참기름을 넣고 한번 더 볶은 후 불을 꺼 줍니다.중 약

In [6]:
# 데이터 로드
feature_num = 2  # 사용할 feature_num 설정
prompt_name = col_name = "relevance"

file_path = f'data/groundTruth/final_groundTruth_f{feature_num}.csv'
df = pd.read_csv(file_path)

# 이벤트 루프 생성 및 실행
loop = asyncio.get_event_loop()
loop.run_until_complete(generate(df, prompt_name, col_name))

# 결과 저장
output_file = f'data/groundTruth/final_groundTruth_f{feature_num}.csv'
df.to_csv(output_file, index=False, encoding='utf-8-sig')
print(f"Processed data saved to {output_file}")

2024-12-23 07:47:53,582 - recipe_logger - INFO - json 출력 파서 초기화 완료.
2024-12-23 07:47:53,583 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 07:47:53,587 - recipe_logger - INFO - LLM 레시피 생성 중...


Processing row 0 (Attempt 1)...


2024-12-23 07:47:54,870 - recipe_logger - INFO - LLM 레시피 생성 완료
2024-12-23 07:47:54,882 - recipe_logger - INFO - json 출력 파서 초기화 완료.
2024-12-23 07:47:54,883 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 07:47:54,884 - recipe_logger - INFO - LLM 레시피 생성 중...


Processing row 1 (Attempt 1)...


2024-12-23 07:47:58,775 - recipe_logger - INFO - LLM 레시피 생성 완료
2024-12-23 07:47:58,785 - recipe_logger - INFO - json 출력 파서 초기화 완료.
2024-12-23 07:47:58,786 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 07:47:58,786 - recipe_logger - INFO - LLM 레시피 생성 중...


...

Processing row 33 (Attempt 1)...


2024-12-23 07:50:09,368 - recipe_logger - INFO - LLM 레시피 생성 완료
2024-12-23 07:50:09,384 - recipe_logger - INFO - json 출력 파서 초기화 완료.
2024-12-23 07:50:09,384 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 07:50:09,385 - recipe_logger - INFO - LLM 레시피 생성 중...


Processing row 34 (Attempt 1)...


2024-12-23 07:50:10,951 - recipe_logger - INFO - LLM 레시피 생성 완료


                                           user_info  \
0  {'user_allergy_ingredients': [], 'user_dislike...   
1  {'user_allergy_ingredients': [], 'user_dislike...   
2  {'user_allergy_ingredients': [], 'user_dislike...   
3  {'user_allergy_ingredients': [], 'user_dislike...   
4  {'user_allergy_ingredients': ['우유', '견과류'], 'u...   

                                         recipe_info  \
0  {'_id': '67610699846f9e5eb975e532', 'title': '...   
1  {'_id': '6761069a846f9e5eb9761021', 'title': '...   
2  {'_id': '6761069a846f9e5eb9760836', 'title': '...   
3  {'_id': '6761069a846f9e5eb9760639', 'title': '...   
4  {'_id': '6761069a846f9e5eb976093b', 'title': '...   

                                          generation  \
0  {'main_changes_from_original_recipe': ['연어 대신 ...   
1  {'main_changes_from_original_recipe': ['🍇 포도주스...   
2  {'main_changes_from_original_recipe': ['양파와 감자...   
3  {'main_changes_from_original_recipe': ['레몬을 굵은...   
4  {'main_changes_from_original_recipe': ['소고기

In [7]:
import pandas as pd
import ast

# 데이터 로드
feature_num = 2  # 사용할 feature_num 설정
prompt_name = col_name = "relevance"

file_path = f'data/groundTruth/final_groundTruth_f{feature_num}.csv'
df = pd.read_csv(file_path)

# 문자열이 아닌 값도 처리하기 위한 함수 정의
def safe_literal_eval(val):
    if isinstance(val, str):
        return ast.literal_eval(val)  # 문자열이면 딕셔너리로 변환
    return val  # 이미 딕셔너리면 그대로 반환

df[col_name] = df[col_name].apply(safe_literal_eval)

# 1. score 평균값 계산
df['score'] = df[col_name].apply(lambda x: x['score'])
score_mean = df['score'].mean()

# 2. reason이 '해당없음'가 아닌 행
df['reason'] = df[col_name].apply(lambda x: x['reason'])
non_x_reasons = df[df['reason'] != '해당없음']

# 3. score 상위 3개의 행
top3_scores = df.nsmallest(10, 'score')

# 결과 출력
print(f"Score 평균값: {score_mean}")

for i in range(len(top3_scores)):
    print(top3_scores.iloc[i].score)
    print(top3_scores.iloc[i].reason)
    print()


Score 평균값: 0.7714285714285715
0.2
The generated content does not align with the user's must-use ingredient (shrimp) and owned ingredients, and the recipe type (drink) may not match the user's typical cooking preferences.

0.3
생성된 내용은 사용자의 요구와 레시피의 관련성을 일부 다루지만, 닭가슴살을 사용하지 않은 이유 외에는 사용자 정보와의 직접적인 관련성이 부족합니다.

0.3
The generated content is relevant to the recipe but does not address the user's specific preferences or available ingredients.

0.4
The generated content fails to include the user's must-use ingredient (asparagus) and does not utilize owned ingredients, reducing relevance, but aligns with the user's cooking level.

0.4
The generated content partially addresses user preferences but fails to include a must-use ingredient and does not adjust the spiciness level.

0.5
The generation partially aligns with the user's spicy level and cooking skill but fails to incorporate must-use ingredients.

0.6
The generated content addresses the exclusion of chicken breast, which is relevant to t

In [8]:
# 데이터 로드
feature_num = 3  # 사용할 feature_num 설정
prompt_name = col_name = "relevance"

file_path = f'data/groundTruth/final_groundTruth_f{feature_num}.csv'
df = pd.read_csv(file_path)

# 이벤트 루프 생성 및 실행
loop = asyncio.get_event_loop()
loop.run_until_complete(generate(df, prompt_name, col_name))

# 결과 저장
output_file = f'data/groundTruth/final_groundTruth_f{feature_num}.csv'
df.to_csv(output_file, index=False, encoding='utf-8-sig')
print(f"Processed data saved to {output_file}")

2024-12-23 07:50:11,019 - recipe_logger - INFO - json 출력 파서 초기화 완료.
2024-12-23 07:50:11,020 - recipe_logger - INFO - langfuse prompt template 생성 완료


2024-12-23 07:50:11,021 - recipe_logger - INFO - LLM 레시피 생성 중...


Processing row 0 (Attempt 1)...


2024-12-23 07:50:15,291 - recipe_logger - INFO - LLM 레시피 생성 완료
2024-12-23 07:50:15,307 - recipe_logger - INFO - json 출력 파서 초기화 완료.
2024-12-23 07:50:15,308 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 07:50:15,309 - recipe_logger - INFO - LLM 레시피 생성 중...


Processing row 1 (Attempt 1)...


...

Processing row 33 (Attempt 1)...


2024-12-23 07:51:56,089 - recipe_logger - INFO - LLM 레시피 생성 완료
2024-12-23 07:51:56,099 - recipe_logger - INFO - json 출력 파서 초기화 완료.
2024-12-23 07:51:56,100 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 07:51:56,100 - recipe_logger - INFO - LLM 레시피 생성 중...


Processing row 34 (Attempt 1)...


2024-12-23 07:51:57,457 - recipe_logger - INFO - LLM 레시피 생성 완료


                                           user_info  \
0  {'user_allergy_ingredients': [], 'user_dislike...   
1  {'user_allergy_ingredients': [], 'user_dislike...   
2  {'user_allergy_ingredients': [], 'user_dislike...   
3  {'user_allergy_ingredients': [], 'user_dislike...   
4  {'user_allergy_ingredients': ['우유', '견과류'], 'u...   

                                         recipe_info  \
0  {'_id': '67610699846f9e5eb975e532', 'title': '...   
1  {'_id': '6761069a846f9e5eb9761021', 'title': '...   
2  {'_id': '6761069a846f9e5eb9760836', 'title': '...   
3  {'_id': '6761069a846f9e5eb9760639', 'title': '...   
4  {'_id': '6761069a846f9e5eb976093b', 'title': '...   

                                          generation  \
0  {'original_recipe_food_group_composition': [{'...   
1  {'original_recipe_food_group_composition': [{'...   
2  {'original_recipe_food_group_composition': [{'...   
3  {'original_recipe_food_group_composition': [{'...   
4  {'original_recipe_food_group_composition': 

In [9]:
import pandas as pd
import ast

# 데이터 로드
feature_num = 3  # 사용할 feature_num 설정
prompt_name = col_name = "relevance"

file_path = f'data/groundTruth/final_groundTruth_f{feature_num}.csv'
df = pd.read_csv(file_path)

# 문자열이 아닌 값도 처리하기 위한 함수 정의
def safe_literal_eval(val):
    if isinstance(val, str):
        return ast.literal_eval(val)  # 문자열이면 딕셔너리로 변환
    return val  # 이미 딕셔너리면 그대로 반환

df[col_name] = df[col_name].apply(safe_literal_eval)

# 1. score 평균값 계산
df['score'] = df[col_name].apply(lambda x: x['score'])
score_mean = df['score'].mean()

# 2. reason이 '해당없음'가 아닌 행
df['reason'] = df[col_name].apply(lambda x: x['reason'])
non_x_reasons = df[df['reason'] != '해당없음']

# 3. score 상위 3개의 행
top3_scores = df.nsmallest(10, 'score')

# 결과 출력
print(f"Score 평균값: {score_mean}")

for i in range(len(top3_scores)):
    print(top3_scores.iloc[i].score)
    print(top3_scores.iloc[i].reason)
    print()


Score 평균값: 0.8528571428571429
0.3
The generated content attempts to incorporate the user's must-use ingredient (kimchi) into the recipe, but the combination of lemon tea and kimchi is unconventional and not directly relevant to the original recipe's purpose or the user's preferences.

0.3
사용자의 요구에 맞춰 새우를 추가하고, 설탕 대신 꿀을 사용한 점은 관련성이 있지만, 레시피의 핵심 맛과 상충되는 사용자의 요구를 무시할 수 있다는 조건을 고려할 때, 전반적으로 관련성이 낮습니다.

0.7
The generated content aligns with the user's preference for spiciness and cooking skill level but fails to incorporate the mandatory use of asparagus.

0.8
The generation effectively incorporates the user's must-use ingredient (chicken breast) and aligns with the user's cooking level and preferences, enhancing the original recipe while maintaining its core characteristics.

0.8
The generated content effectively incorporates the user's must-use ingredient (mushroom) and aligns with the user's cooking level and available ingredients, making it relevant to the user's needs.

0.85
생성된 내용은 사용

### 2. eval prompt로 test - relevance
- 1,2,3번 모두 0.0

In [5]:
from langchain_teddynote import logging
# set_enable=False 로 지정하면 추적을 하지 않습니다.
logging.langsmith("랭체인 튜토리얼 프로젝트", set_enable=False)

import nest_asyncio
import asyncio
import pandas as pd
import sys
sys.path.append('../')  # 상위 디렉토리의 src 폴더를 경로에 추가
from src.recipe_change_origin import eval_recipe

# nest_asyncio로 이미 실행 중인 루프에서 중첩 실행 허용
nest_asyncio.apply()

# 동시 요청 제한과 재시도 설정
MAX_CONCURRENT_REQUESTS = 3
RETRY_LIMIT = 3

async def generate(df, prompt_name, col_name):
    df[col_name] = None  # 결과 저장 열 생성

    # Semaphore 생성
    semaphore = asyncio.Semaphore(MAX_CONCURRENT_REQUESTS)

    tasks = []  # 모든 작업을 저장할 리스트

    for i in range(len(df)):
        retry_count = 0
        while retry_count <= RETRY_LIMIT:
            try:
                # Semaphore로 동시 요청 제한
                async with semaphore:
                    print(f"Processing row {i} (Attempt {retry_count + 1})...")
                    result = await eval_recipe(df.iloc[i].recipe_info, df.iloc[i].user_info, df.iloc[i].generation, df.iloc[i].groundTruth, prompt_name)
                    df.at[i, col_name] = result  # 결과 저장
                    break  # 성공하면 반복문 종료
            except Exception as e:
                retry_count += 1
                if retry_count > RETRY_LIMIT:
                    print(f"Failed to process row {i} after {RETRY_LIMIT} retries.")
                    break  # 재시도 초과 시 반복문 종료
                print(f"Error processing row {i}: {e}. Retrying...")
                await asyncio.sleep(10 ** retry_count)  # 재시도 전 대기

        tasks.append(asyncio.sleep(0))  # 리스트에 임시 작업 추가 (추후 확장 가능)

    await asyncio.gather(*tasks)  # 모든 작업 실행

    # 결과 확인
    print(df.head())
    


LangSmith 추적을 하지 않습니다.


In [6]:
# 데이터 로드
feature_num = 1  # 사용할 feature_num 설정
prompt_name = col_name = "relevance_recipe"

file_path = f'data/groundTruth/final_groundTruth_f{feature_num}.csv'
df = pd.read_csv(file_path)

# 이벤트 루프 생성 및 실행
loop = asyncio.get_event_loop()
loop.run_until_complete(generate(df, prompt_name, col_name))

# 결과 저장
output_file = f'data/groundTruth/final_groundTruth_f{feature_num}.csv'
df.to_csv(output_file, index=False, encoding='utf-8-sig')
print(f"Processed data saved to {output_file}")

2024-12-23 08:21:48,830 - recipe_logger - INFO - json 출력 파서 초기화 완료.


Processing row 0 (Attempt 1)...


2024-12-23 08:21:49,374 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 08:21:49,377 - recipe_logger - INFO - LLM 레시피 생성 중...
2024-12-23 08:21:52,663 - recipe_logger - INFO - LLM 레시피 생성 완료
2024-12-23 08:21:52,675 - recipe_logger - INFO - json 출력 파서 초기화 완료.
2024-12-23 08:21:52,675 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 08:21:52,676 - recipe_logger - INFO - LLM 레시피 생성 중...


Processing row 1 (Attempt 1)...


2024-12-23 08:21:57,258 - recipe_logger - INFO - LLM 레시피 생성 완료
2024-12-23 08:21:57,268 - recipe_logger - INFO - json 출력 파서 초기화 완료.
2024-12-23 08:21:57,269 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 08:21:57,270 - recipe_logger - INFO - LLM 레시피 생성 중...


...

Processing row 33 (Attempt 1)...


2024-12-23 08:24:08,430 - recipe_logger - INFO - LLM 레시피 생성 완료
2024-12-23 08:24:08,441 - recipe_logger - INFO - json 출력 파서 초기화 완료.
2024-12-23 08:24:08,441 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 08:24:08,442 - recipe_logger - INFO - LLM 레시피 생성 중...


Processing row 34 (Attempt 1)...


2024-12-23 08:24:14,087 - recipe_logger - INFO - LLM 레시피 생성 완료


                                           user_info  \
0  {'user_allergy_ingredients': [], 'user_dislike...   
1  {'user_allergy_ingredients': [], 'user_dislike...   
2  {'user_allergy_ingredients': [], 'user_dislike...   
3  {'user_allergy_ingredients': [], 'user_dislike...   
4  {'user_allergy_ingredients': ['우유', '견과류'], 'u...   

                                         recipe_info  \
0  {'_id': '67610699846f9e5eb975e532', 'title': '...   
1  {'_id': '6761069a846f9e5eb9761021', 'title': '...   
2  {'_id': '6761069a846f9e5eb9760836', 'title': '...   
3  {'_id': '6761069a846f9e5eb9760639', 'title': '...   
4  {'_id': '6761069a846f9e5eb976093b', 'title': '...   

                                          generation  \
0  {'main_changes_from_original_recipe': ['🥗 닭가슴살...   
1  {'main_changes_from_original_recipe': ['🍇 포도주스...   
2  {'main_changes_from_original_recipe': ['🥣 양파는 ...   
3  {'main_changes_from_original_recipe': ['🍋 레몬을 ...   
4  {'main_changes_from_original_recipe': ['🥣 황

In [7]:
import pandas as pd
import ast

# 데이터 로드
feature_num = 1  # 사용할 feature_num 설정
prompt_name = col_name = "relevance_recipe"

file_path = f'data/groundTruth/final_groundTruth_f{feature_num}.csv'
df = pd.read_csv(file_path)

# 문자열이 아닌 값도 처리하기 위한 함수 정의
def safe_literal_eval(val):
    if isinstance(val, str):
        return ast.literal_eval(val)  # 문자열이면 딕셔너리로 변환
    return val  # 이미 딕셔너리면 그대로 반환

df[col_name] = df[col_name].apply(safe_literal_eval)

# 1. score 평균값 계산
df['score'] = df[col_name].apply(lambda x: x['score'])
score_mean = df['score'].mean()

# 2. reason이 '해당없음'가 아닌 행
df['reason'] = df[col_name].apply(lambda x: x['reason'])
non_x_reasons = df[df['reason'] != '해당없음']

# 3. score 상위 3개의 행
top3_scores = df.nsmallest(10, 'score')

# 결과 출력
print(f"Score 평균값: {score_mean}")

for i in range(len(top3_scores)):
    print(top3_scores.iloc[i].user_info)
    print(top3_scores.iloc[i].recipe_info)
    print(top3_scores.iloc[i].generation)
    print(top3_scores.iloc[i].score)
    print(top3_scores.iloc[i].reason)
    print()


Score 평균값: 0.817142857142857
{'user_allergy_ingredients': ['우유', '크림', '버터'], 'user_dislike_ingredients': ['소고기', '양고기'], 'user_spicy_level': '4단계', 'user_cooking_level': '중급', 'user_owned_ingredients': ['양파', '새우', '파스타면', '방울토마토', '바질'], 'user_basic_seasoning': ['허브솔트', '올리브유', '레몬즙', '후추', '바질페스토'], 'must_use_ingredients': ['새우']}
{'_id': '6761069a846f9e5eb97606ed', 'title': '로시:) 식혜 만드는법 살얼음동동 밥알동동!', 'type_key': '차/음료/술', 'method_key': '끓이기', 'servings': '4인분', 'cooking_time': '2시간 이상', 'difficulty': '중급', 'ingredients': ['물(3~4L)', '엿기름(200g)', '쌀(1.5컵)', '설탕(3컵)', '생강(1/2톨)'], 'cooking_steps': ['분량의 물에 베보자기나 면보로 감싼 엿기름을 주물러 물을 빼줍니다뽀얗게 금방 나오지만 진하게 나오도록5분가량은 꾹꾹 눌러가며 뽑아내 해주세요보자기에 담긴 엿기름은 버리고우려낸물에서 하얀 찌꺼기가 가라앉도록 기다립니다.', '이 시간에 쌀1컵반으로 고두밥을 지어 두세요!아래 하얀찌꺼기들이 가라앉아야하니자꾸 기울여 확인하지 마시고 3~4시간 그냥 두세요', '고두밥에 윗물을 부어 줍니다최대한 하얀찌거기가들어가지 않게 부어보세요', '8시간 후에 밥알이 삭아서 떠있는 모습이예요밥알이 20~30개 뜬다면 삭히는 시간은 좀 줄여도 괜찮습니다', '생강을 반톨정도 설탕 2~3컵 입맛에 맞게 넣어 부글부글 끓입니다', '끓으면서 올라오는 거품과 이물질을 걷어주시면 좋아요', '밥알이 동동뜨는 식혜를 드

In [8]:
# 데이터 로드
feature_num = 2  # 사용할 feature_num 설정
prompt_name = col_name = "relevance_recipe"

file_path = f'data/groundTruth/final_groundTruth_f{feature_num}.csv'
df = pd.read_csv(file_path)

# 이벤트 루프 생성 및 실행
loop = asyncio.get_event_loop()
loop.run_until_complete(generate(df, prompt_name, col_name))

# 결과 저장
output_file = f'data/groundTruth/final_groundTruth_f{feature_num}.csv'
df.to_csv(output_file, index=False, encoding='utf-8-sig')
print(f"Processed data saved to {output_file}")

2024-12-23 08:25:48,896 - recipe_logger - INFO - json 출력 파서 초기화 완료.
2024-12-23 08:25:48,897 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 08:25:48,899 - recipe_logger - INFO - LLM 레시피 생성 중...


Processing row 0 (Attempt 1)...


2024-12-23 08:25:53,315 - recipe_logger - INFO - LLM 레시피 생성 완료
2024-12-23 08:25:53,326 - recipe_logger - INFO - json 출력 파서 초기화 완료.
2024-12-23 08:25:53,326 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 08:25:53,327 - recipe_logger - INFO - LLM 레시피 생성 중...


Processing row 1 (Attempt 1)...


2024-12-23 08:25:56,707 - recipe_logger - INFO - LLM 레시피 생성 완료
2024-12-23 08:25:56,719 - recipe_logger - INFO - json 출력 파서 초기화 완료.
2024-12-23 08:25:56,720 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 08:25:56,721 - recipe_logger - INFO - LLM 레시피 생성 중...


...

Processing row 33 (Attempt 1)...


2024-12-23 08:28:20,075 - recipe_logger - INFO - LLM 레시피 생성 완료
2024-12-23 08:28:20,086 - recipe_logger - INFO - json 출력 파서 초기화 완료.
2024-12-23 08:28:20,086 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 08:28:20,087 - recipe_logger - INFO - LLM 레시피 생성 중...


Processing row 34 (Attempt 1)...


2024-12-23 08:28:26,721 - recipe_logger - INFO - LLM 레시피 생성 완료


                                           user_info  \
0  {'user_allergy_ingredients': [], 'user_dislike...   
1  {'user_allergy_ingredients': [], 'user_dislike...   
2  {'user_allergy_ingredients': [], 'user_dislike...   
3  {'user_allergy_ingredients': [], 'user_dislike...   
4  {'user_allergy_ingredients': ['우유', '견과류'], 'u...   

                                         recipe_info  \
0  {'_id': '67610699846f9e5eb975e532', 'title': '...   
1  {'_id': '6761069a846f9e5eb9761021', 'title': '...   
2  {'_id': '6761069a846f9e5eb9760836', 'title': '...   
3  {'_id': '6761069a846f9e5eb9760639', 'title': '...   
4  {'_id': '6761069a846f9e5eb976093b', 'title': '...   

                                          generation  \
0  {'main_changes_from_original_recipe': ['연어 대신 ...   
1  {'main_changes_from_original_recipe': ['🍇 포도주스...   
2  {'main_changes_from_original_recipe': ['양파와 감자...   
3  {'main_changes_from_original_recipe': ['레몬을 굵은...   
4  {'main_changes_from_original_recipe': ['소고기

In [9]:
import pandas as pd
import ast

# 데이터 로드
feature_num = 2  # 사용할 feature_num 설정
prompt_name = col_name = "relevance_recipe"

file_path = f'data/groundTruth/final_groundTruth_f{feature_num}.csv'
df = pd.read_csv(file_path)

# 문자열이 아닌 값도 처리하기 위한 함수 정의
def safe_literal_eval(val):
    if isinstance(val, str):
        return ast.literal_eval(val)  # 문자열이면 딕셔너리로 변환
    return val  # 이미 딕셔너리면 그대로 반환

df[col_name] = df[col_name].apply(safe_literal_eval)

# 1. score 평균값 계산
df['score'] = df[col_name].apply(lambda x: x['score'])
score_mean = df['score'].mean()

# 2. reason이 '해당없음'가 아닌 행
df['reason'] = df[col_name].apply(lambda x: x['reason'])
non_x_reasons = df[df['reason'] != '해당없음']

# 3. score 상위 3개의 행
top3_scores = df.nsmallest(10, 'score')

# 결과 출력
print(f"Score 평균값: {score_mean}")

for i in range(len(top3_scores)):
    print(top3_scores.iloc[i].user_info)
    print(top3_scores.iloc[i].recipe_info)
    print(top3_scores.iloc[i].generation)
    print(top3_scores.iloc[i].score)
    print(top3_scores.iloc[i].reason)
    print()


Score 평균값: 0.8300000000000001
{'user_allergy_ingredients': ['새우', '게', '견과류'], 'user_dislike_ingredients': ['닭고기', '브로콜리', '오이'], 'user_spicy_level': '5단계', 'user_cooking_level': '고급', 'user_owned_ingredients': ['연어', '아스파라거스', '파스타면'], 'user_basic_seasoning': ['버터', '허브솔트', '파마산 치즈'], 'must_use_ingredients': ['아스파라거스']}
{'_id': '6761069a846f9e5eb97600e1', 'title': '계란초 만들기 핑거푸드요리 완숙반숙 삶는시간', 'type_key': '반찬', 'method_key': '삶기', 'servings': '2인분', 'cooking_time': '5분 이내', 'difficulty': '초급', 'ingredients': ['달걀(2개)', '초장(2t)', '식초(약간)', '소금(약간)'], 'cooking_steps': ['계란을 삶기만 해서 새콤달콤한 초장만 찍어서 먹거나뿌려서 먹음 된는 계란초', '물에 계란을 넣고식초와 소금을 약간 넣어 주세요.소금은 껍질도 잘 까지게 해주고요식초는 물이 끓면서 계란이 서로 부딪히면서내용물이 나와서 계란국이 되지 않게 되지요.계란반숙삶는법 #반숙삶는시간 저는 거의 7분완숙 삶는법 거의 11분 삶아요.', '찬물에 헹구어 껍지을 까서 적당한 크기로 잘라 주세요.', '새콤달콤한 초장만 뿌려주면 간단하게밥반찬으로도 손색이 없는 계란초만드는법 완성이랍니다.'], 'tips': []}
{'main_changes_from_original_recipe': ['아스파라거스는 계란초와 잘 어울리지 않아서 🥚❌ 빼버렸어요! 초장과 계란의 조화만으로도 충분히 맛있답니다! 🍳✨', '조리 과정도 간단하게 줄여서 4단계로 만들었어요! ⏳💨'], 'reas

In [10]:
# 데이터 로드
feature_num = 3  # 사용할 feature_num 설정
prompt_name = col_name = "relevance_recipe"

file_path = f'data/groundTruth/final_groundTruth_f{feature_num}.csv'
df = pd.read_csv(file_path)

# 이벤트 루프 생성 및 실행
loop = asyncio.get_event_loop()
loop.run_until_complete(generate(df, prompt_name, col_name))

# 결과 저장
output_file = f'data/groundTruth/final_groundTruth_f{feature_num}.csv'
df.to_csv(output_file, index=False, encoding='utf-8-sig')
print(f"Processed data saved to {output_file}")

2024-12-23 08:28:26,779 - recipe_logger - INFO - json 출력 파서 초기화 완료.
2024-12-23 08:28:26,780 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 08:28:26,781 - recipe_logger - INFO - LLM 레시피 생성 중...


Processing row 0 (Attempt 1)...


2024-12-23 08:28:30,272 - recipe_logger - INFO - LLM 레시피 생성 완료
2024-12-23 08:28:30,288 - recipe_logger - INFO - json 출력 파서 초기화 완료.
2024-12-23 08:28:30,289 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 08:28:30,290 - recipe_logger - INFO - LLM 레시피 생성 중...


Processing row 1 (Attempt 1)...


2024-12-23 08:28:31,604 - recipe_logger - INFO - LLM 레시피 생성 완료
2024-12-23 08:28:31,616 - recipe_logger - INFO - json 출력 파서 초기화 완료.
2024-12-23 08:28:31,616 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 08:28:31,617 - recipe_logger - INFO - LLM 레시피 생성 중...


...

Processing row 33 (Attempt 1)...


2024-12-23 08:30:21,221 - recipe_logger - INFO - LLM 레시피 생성 완료
2024-12-23 08:30:21,232 - recipe_logger - INFO - json 출력 파서 초기화 완료.
2024-12-23 08:30:21,232 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 08:30:21,233 - recipe_logger - INFO - LLM 레시피 생성 중...


Processing row 34 (Attempt 1)...


2024-12-23 08:30:25,916 - recipe_logger - INFO - LLM 레시피 생성 완료


                                           user_info  \
0  {'user_allergy_ingredients': [], 'user_dislike...   
1  {'user_allergy_ingredients': [], 'user_dislike...   
2  {'user_allergy_ingredients': [], 'user_dislike...   
3  {'user_allergy_ingredients': [], 'user_dislike...   
4  {'user_allergy_ingredients': ['우유', '견과류'], 'u...   

                                         recipe_info  \
0  {'_id': '67610699846f9e5eb975e532', 'title': '...   
1  {'_id': '6761069a846f9e5eb9761021', 'title': '...   
2  {'_id': '6761069a846f9e5eb9760836', 'title': '...   
3  {'_id': '6761069a846f9e5eb9760639', 'title': '...   
4  {'_id': '6761069a846f9e5eb976093b', 'title': '...   

                                          generation  \
0  {'original_recipe_food_group_composition': [{'...   
1  {'original_recipe_food_group_composition': [{'...   
2  {'original_recipe_food_group_composition': [{'...   
3  {'original_recipe_food_group_composition': [{'...   
4  {'original_recipe_food_group_composition': 

In [11]:
import pandas as pd
import ast

# 데이터 로드
feature_num = 3  # 사용할 feature_num 설정
prompt_name = col_name = "relevance_recipe"

file_path = f'data/groundTruth/final_groundTruth_f{feature_num}.csv'
df = pd.read_csv(file_path)

# 문자열이 아닌 값도 처리하기 위한 함수 정의
def safe_literal_eval(val):
    if isinstance(val, str):
        return ast.literal_eval(val)  # 문자열이면 딕셔너리로 변환
    return val  # 이미 딕셔너리면 그대로 반환

df[col_name] = df[col_name].apply(safe_literal_eval)

# 1. score 평균값 계산
df['score'] = df[col_name].apply(lambda x: x['score'])
score_mean = df['score'].mean()

# 2. reason이 '해당없음'가 아닌 행
df['reason'] = df[col_name].apply(lambda x: x['reason'])
non_x_reasons = df[df['reason'] != '해당없음']

# 3. score 상위 3개의 행
top3_scores = df.nsmallest(10, 'score')

# 결과 출력
print(f"Score 평균값: {score_mean}")

for i in range(len(top3_scores)):
    print(top3_scores.iloc[i].user_info)
    print(top3_scores.iloc[i].recipe_info)
    print(top3_scores.iloc[i].generation)
    print(top3_scores.iloc[i].score)
    print(top3_scores.iloc[i].reason)
    print()


Score 평균값: 0.8557142857142856
{'user_allergy_ingredients': [], 'user_dislike_ingredients': ['돈가스', '치킨 너겟', '감자튀김', '고로케', '새우튀김'], 'user_spicy_level': '2단계', 'user_cooking_level': '초급', 'user_owned_ingredients': ['닭가슴살', '단호박', '아보카도'], 'user_basic_seasoning': ['후추', '올리브유', '발사믹 식초'], 'must_use_ingredients': ['닭가슴살']}
{'_id': '67610699846f9e5eb975e745', 'title': '바나나미숫가루쉐이크', 'type_key': '차/음료/술', 'method_key': '갈기', 'servings': '1인분', 'cooking_time': '', 'difficulty': '', 'ingredients': ['미숫가루 15g', '바나나 50g', '우유 100ml'], 'cooking_steps': ['1. 바나나 껍질을 벗긴다.', '2. 바나나를 알맞은 크기로 잘라 준비해 둔다.', '3. 우유와 미숫가루를 알맞은 양으로 계량한다.', '4. ③과 바나나를 믹서기에 넣는다.', '5. 건더기가 생기지 않을 정도로 간다.', '6. 컵에 담아 완성시킨다.'], 'tips': ['바나나는 칼륨 함량이 풍부하여 나트륨 배출을 도와요.']}
{'original_recipe_food_group_composition': [{'food_group': '곡류', 'amount': 0.05}, {'food_group': '우유·유제품류', 'amount': 0.5}, {'food_group': '과일류', 'amount': 0.5}], 'user_meal_food_group_requirements': [{'food_group': '고기·생선·달걀·콩류', 'amount': 1}, {'food_group'

### relevance_user

In [12]:
# 데이터 로드
feature_num = 1  # 사용할 feature_num 설정
prompt_name = col_name = "relevance_user"

file_path = f'data/groundTruth/final_groundTruth_f{feature_num}.csv'
df = pd.read_csv(file_path)

# 이벤트 루프 생성 및 실행
loop = asyncio.get_event_loop()
loop.run_until_complete(generate(df, prompt_name, col_name))

# 결과 저장
output_file = f'data/groundTruth/final_groundTruth_f{feature_num}.csv'
df.to_csv(output_file, index=False, encoding='utf-8-sig')
print(f"Processed data saved to {output_file}")

2024-12-23 08:32:08,023 - recipe_logger - INFO - json 출력 파서 초기화 완료.


Processing row 0 (Attempt 1)...


2024-12-23 08:32:08,536 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 08:32:08,539 - recipe_logger - INFO - LLM 레시피 생성 중...
2024-12-23 08:32:13,725 - recipe_logger - INFO - LLM 레시피 생성 완료
2024-12-23 08:32:13,739 - recipe_logger - INFO - json 출력 파서 초기화 완료.
2024-12-23 08:32:13,740 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 08:32:13,741 - recipe_logger - INFO - LLM 레시피 생성 중...


Processing row 1 (Attempt 1)...


2024-12-23 08:32:19,141 - recipe_logger - INFO - LLM 레시피 생성 완료
2024-12-23 08:32:19,152 - recipe_logger - INFO - json 출력 파서 초기화 완료.
2024-12-23 08:32:19,154 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 08:32:19,154 - recipe_logger - INFO - LLM 레시피 생성 중...


...

Processing row 33 (Attempt 1)...


2024-12-23 08:35:07,722 - recipe_logger - INFO - LLM 레시피 생성 완료
2024-12-23 08:35:07,734 - recipe_logger - INFO - json 출력 파서 초기화 완료.
2024-12-23 08:35:07,735 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 08:35:07,735 - recipe_logger - INFO - LLM 레시피 생성 중...


Processing row 34 (Attempt 1)...


2024-12-23 08:35:14,418 - recipe_logger - INFO - LLM 레시피 생성 완료


                                           user_info  \
0  {'user_allergy_ingredients': [], 'user_dislike...   
1  {'user_allergy_ingredients': [], 'user_dislike...   
2  {'user_allergy_ingredients': [], 'user_dislike...   
3  {'user_allergy_ingredients': [], 'user_dislike...   
4  {'user_allergy_ingredients': ['우유', '견과류'], 'u...   

                                         recipe_info  \
0  {'_id': '67610699846f9e5eb975e532', 'title': '...   
1  {'_id': '6761069a846f9e5eb9761021', 'title': '...   
2  {'_id': '6761069a846f9e5eb9760836', 'title': '...   
3  {'_id': '6761069a846f9e5eb9760639', 'title': '...   
4  {'_id': '6761069a846f9e5eb976093b', 'title': '...   

                                          generation  \
0  {'main_changes_from_original_recipe': ['🥗 닭가슴살...   
1  {'main_changes_from_original_recipe': ['🍇 포도주스...   
2  {'main_changes_from_original_recipe': ['🥣 양파는 ...   
3  {'main_changes_from_original_recipe': ['🍋 레몬을 ...   
4  {'main_changes_from_original_recipe': ['🥣 황

In [13]:
import pandas as pd
import ast

# 데이터 로드
feature_num = 1  # 사용할 feature_num 설정
prompt_name = col_name = "relevance_user"

file_path = f'data/groundTruth/final_groundTruth_f{feature_num}.csv'
df = pd.read_csv(file_path)

# 문자열이 아닌 값도 처리하기 위한 함수 정의
def safe_literal_eval(val):
    if isinstance(val, str):
        return ast.literal_eval(val)  # 문자열이면 딕셔너리로 변환
    return val  # 이미 딕셔너리면 그대로 반환

df[col_name] = df[col_name].apply(safe_literal_eval)

# 1. score 평균값 계산
df['score'] = df[col_name].apply(lambda x: x['score'])
score_mean = df['score'].mean()

# 2. reason이 '해당없음'가 아닌 행
df['reason'] = df[col_name].apply(lambda x: x['reason'])
non_x_reasons = df[df['reason'] != '해당없음']

# 3. score 상위 3개의 행
top3_scores = df.nsmallest(10, 'score')

# 결과 출력
print(f"Score 평균값: {score_mean}")

for i in range(len(top3_scores)):
    print(top3_scores.iloc[i].user_info)
    print(top3_scores.iloc[i].recipe_info)
    print(top3_scores.iloc[i].generation)
    print(top3_scores.iloc[i].score)
    print(top3_scores.iloc[i].reason)
    print()


Score 평균값: 0.8728571428571428
{'user_allergy_ingredients': [], 'user_dislike_ingredients': [], 'user_spicy_level': '3단계', 'user_cooking_level': '중급', 'user_owned_ingredients': ['닭가슴살', '두부', '브로콜리'], 'user_basic_seasoning': ['소금', '후추', '올리브유'], 'must_use_ingredients': ['닭가슴살']}
{'_id': '6761069a846f9e5eb9761fe5', 'title': '저어서 만들어야 하는 수플레오믈렛 만들기', 'type_key': '퓨전', 'method_key': '굽기', 'servings': '2인분', 'cooking_time': '30분 이내', 'difficulty': '중급', 'ingredients': ['달걀(3개)', '생크림(4숟가락)', '설탕(4숟가락)', '버터(약간)', '메이플시럽(적당량)', '과일(적당량)'], 'cooking_steps': ['달걀은 흰자와 노른자를 분리한다.', '노른자를 살짝 풀어 준 후 생크림을 넣고 살짝 거품이 올라올 정도로 휘핑한다.', '흰자에 설탕을 넣어가며 휘핑하여 머랭을 만든다.', '휘핑 한 노른자에 머랭 1/2분량을 넣고 섞어 반죽을 만든다.', '달군 팬에 버터를 녹인 후 반죽을 올려 약 불로 5분간 익힌다.', '밑면과 가장자리가 노릇해지면 남겨둔 머랭을 반만 올린 후 반으로 접는다.', '뚜껑을 덮고 약 불에서 5분간 더 익힌다.', '접시에 수플레 오믈렛을 담고 메이플 시럽, 과일을 곁들여 완성한다'], 'tips': []}
{'main_changes_from_original_recipe': ['🥣 닭가슴살을 부드러운 생크림으로 대체해요! 🍶', '🍳 달걀 흰자와 노른자를 분리한 후, 생크림을 넣고 휘핑해 부드러운 질감을 만들어줘요! 🎈', '🔥 팬에 반죽을 올리고 약한 불

In [14]:
# 데이터 로드
feature_num = 2  # 사용할 feature_num 설정
prompt_name = col_name = "relevance_user"

file_path = f'data/groundTruth/final_groundTruth_f{feature_num}.csv'
df = pd.read_csv(file_path)

# 이벤트 루프 생성 및 실행
loop = asyncio.get_event_loop()
loop.run_until_complete(generate(df, prompt_name, col_name))

# 결과 저장
output_file = f'data/groundTruth/final_groundTruth_f{feature_num}.csv'
df.to_csv(output_file, index=False, encoding='utf-8-sig')
print(f"Processed data saved to {output_file}")

2024-12-23 08:35:14,503 - recipe_logger - INFO - json 출력 파서 초기화 완료.
2024-12-23 08:35:14,504 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 08:35:14,506 - recipe_logger - INFO - LLM 레시피 생성 중...


Processing row 0 (Attempt 1)...


2024-12-23 08:35:19,713 - recipe_logger - INFO - LLM 레시피 생성 완료
2024-12-23 08:35:19,724 - recipe_logger - INFO - json 출력 파서 초기화 완료.
2024-12-23 08:35:19,724 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 08:35:19,725 - recipe_logger - INFO - LLM 레시피 생성 중...


Processing row 1 (Attempt 1)...


2024-12-23 08:35:25,321 - recipe_logger - INFO - LLM 레시피 생성 완료
2024-12-23 08:35:25,333 - recipe_logger - INFO - json 출력 파서 초기화 완료.
2024-12-23 08:35:25,333 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 08:35:25,334 - recipe_logger - INFO - LLM 레시피 생성 중...


...

Processing row 33 (Attempt 1)...


2024-12-23 08:38:19,277 - recipe_logger - INFO - LLM 레시피 생성 완료
2024-12-23 08:38:19,289 - recipe_logger - INFO - json 출력 파서 초기화 완료.
2024-12-23 08:38:19,290 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 08:38:19,290 - recipe_logger - INFO - LLM 레시피 생성 중...


Processing row 34 (Attempt 1)...


2024-12-23 08:38:22,791 - recipe_logger - INFO - LLM 레시피 생성 완료


                                           user_info  \
0  {'user_allergy_ingredients': [], 'user_dislike...   
1  {'user_allergy_ingredients': [], 'user_dislike...   
2  {'user_allergy_ingredients': [], 'user_dislike...   
3  {'user_allergy_ingredients': [], 'user_dislike...   
4  {'user_allergy_ingredients': ['우유', '견과류'], 'u...   

                                         recipe_info  \
0  {'_id': '67610699846f9e5eb975e532', 'title': '...   
1  {'_id': '6761069a846f9e5eb9761021', 'title': '...   
2  {'_id': '6761069a846f9e5eb9760836', 'title': '...   
3  {'_id': '6761069a846f9e5eb9760639', 'title': '...   
4  {'_id': '6761069a846f9e5eb976093b', 'title': '...   

                                          generation  \
0  {'main_changes_from_original_recipe': ['연어 대신 ...   
1  {'main_changes_from_original_recipe': ['🍇 포도주스...   
2  {'main_changes_from_original_recipe': ['양파와 감자...   
3  {'main_changes_from_original_recipe': ['레몬을 굵은...   
4  {'main_changes_from_original_recipe': ['소고기

In [15]:
import pandas as pd
import ast

# 데이터 로드
feature_num = 2  # 사용할 feature_num 설정
prompt_name = col_name = "relevance_user"

file_path = f'data/groundTruth/final_groundTruth_f{feature_num}.csv'
df = pd.read_csv(file_path)

# 문자열이 아닌 값도 처리하기 위한 함수 정의
def safe_literal_eval(val):
    if isinstance(val, str):
        return ast.literal_eval(val)  # 문자열이면 딕셔너리로 변환
    return val  # 이미 딕셔너리면 그대로 반환

df[col_name] = df[col_name].apply(safe_literal_eval)

# 1. score 평균값 계산
df['score'] = df[col_name].apply(lambda x: x['score'])
score_mean = df['score'].mean()

# 2. reason이 '해당없음'가 아닌 행
df['reason'] = df[col_name].apply(lambda x: x['reason'])
non_x_reasons = df[df['reason'] != '해당없음']

# 3. score 상위 3개의 행
top3_scores = df.nsmallest(10, 'score')

# 결과 출력
print(f"Score 평균값: {score_mean}")

for i in range(len(top3_scores)):
    print(top3_scores.iloc[i].user_info)
    print(top3_scores.iloc[i].recipe_info)
    print(top3_scores.iloc[i].generation)
    print(top3_scores.iloc[i].score)
    print(top3_scores.iloc[i].reason)
    print()


Score 평균값: 0.897142857142857
{'user_allergy_ingredients': [], 'user_dislike_ingredients': [], 'user_spicy_level': '3단계', 'user_cooking_level': '중급', 'user_owned_ingredients': ['닭가슴살', '두부', '브로콜리'], 'user_basic_seasoning': ['소금', '후추', '올리브유'], 'must_use_ingredients': ['닭가슴살']}
{'_id': '67610699846f9e5eb975e532', 'title': '연어샐러드', 'type_key': '샐러드', 'method_key': '회', 'servings': '1인분', 'cooking_time': '', 'difficulty': '', 'ingredients': ['연어(150g)', '레몬(1/4개)', '발사믹식초(50g)', '어린잎채소(30g)', '후춧가루(0.01g)', '올리브오일(20g)'], 'cooking_steps': ['1. 연어는 깍둑썰기한다.', '2. 썰어 놓은 연어는 후춧가루와 레몬으로 마리네이드한다.', '3. 어린잎은 찬물에 담궈둔다.', '4. 담궈 놓은 어린잎을 체에 받쳐 물기를 뺀다.', '5. 레몬과 올리브오일을 섞는다.', '6. ?번에 발사믹소스를 넣고 연어 샐러드 양념을 만들고, 접시에 연어와 물기를 뺀 어린잎을 담는다.'], 'tips': ['발사믹소스에 레몬과 올리브오일을 섞어 별도의 소금을 넣지 않을 수 있어요.']}
{'main_changes_from_original_recipe': ['연어 대신 닭가슴살을 사용하지 않고, 연어를 그대로 유지하여 고소한 맛을 살렸어요! 🐟✨', '조리 과정에서 어린잎채소를 물에 담궈두는 과정을 간소화했어요. 💧🥗', '발사믹식초와 레몬을 섞어 소스를 만드는 과정을 간단하게 정리했어요! 🍋🥄'], 'reason_for_changes': ['닭가슴살은 연어의 

In [16]:
# 데이터 로드
feature_num = 3  # 사용할 feature_num 설정
prompt_name = col_name = "relevance_user"

file_path = f'data/groundTruth/final_groundTruth_f{feature_num}.csv'
df = pd.read_csv(file_path)

# 이벤트 루프 생성 및 실행
loop = asyncio.get_event_loop()
loop.run_until_complete(generate(df, prompt_name, col_name))

# 결과 저장
output_file = f'data/groundTruth/final_groundTruth_f{feature_num}.csv'
df.to_csv(output_file, index=False, encoding='utf-8-sig')
print(f"Processed data saved to {output_file}")

2024-12-23 08:38:22,850 - recipe_logger - INFO - json 출력 파서 초기화 완료.
2024-12-23 08:38:22,850 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 08:38:22,851 - recipe_logger - INFO - LLM 레시피 생성 중...


Processing row 0 (Attempt 1)...


2024-12-23 08:38:28,837 - recipe_logger - INFO - LLM 레시피 생성 완료
2024-12-23 08:38:28,848 - recipe_logger - INFO - json 출력 파서 초기화 완료.
2024-12-23 08:38:28,848 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 08:38:28,849 - recipe_logger - INFO - LLM 레시피 생성 중...


Processing row 1 (Attempt 1)...


2024-12-23 08:38:34,556 - recipe_logger - INFO - LLM 레시피 생성 완료
2024-12-23 08:38:34,566 - recipe_logger - INFO - json 출력 파서 초기화 완료.
2024-12-23 08:38:34,566 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 08:38:34,567 - recipe_logger - INFO - LLM 레시피 생성 중...


...

Processing row 33 (Attempt 1)...


2024-12-23 08:41:19,844 - recipe_logger - INFO - LLM 레시피 생성 완료
2024-12-23 08:41:19,854 - recipe_logger - INFO - json 출력 파서 초기화 완료.
2024-12-23 08:41:19,855 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 08:41:19,855 - recipe_logger - INFO - LLM 레시피 생성 중...


Processing row 34 (Attempt 1)...


2024-12-23 08:41:26,403 - recipe_logger - INFO - LLM 레시피 생성 완료


                                           user_info  \
0  {'user_allergy_ingredients': [], 'user_dislike...   
1  {'user_allergy_ingredients': [], 'user_dislike...   
2  {'user_allergy_ingredients': [], 'user_dislike...   
3  {'user_allergy_ingredients': [], 'user_dislike...   
4  {'user_allergy_ingredients': ['우유', '견과류'], 'u...   

                                         recipe_info  \
0  {'_id': '67610699846f9e5eb975e532', 'title': '...   
1  {'_id': '6761069a846f9e5eb9761021', 'title': '...   
2  {'_id': '6761069a846f9e5eb9760836', 'title': '...   
3  {'_id': '6761069a846f9e5eb9760639', 'title': '...   
4  {'_id': '6761069a846f9e5eb976093b', 'title': '...   

                                          generation  \
0  {'original_recipe_food_group_composition': [{'...   
1  {'original_recipe_food_group_composition': [{'...   
2  {'original_recipe_food_group_composition': [{'...   
3  {'original_recipe_food_group_composition': [{'...   
4  {'original_recipe_food_group_composition': 

In [17]:
import pandas as pd
import ast

# 데이터 로드
feature_num = 3  # 사용할 feature_num 설정
prompt_name = col_name = "relevance_user"

file_path = f'data/groundTruth/final_groundTruth_f{feature_num}.csv'
df = pd.read_csv(file_path)

# 문자열이 아닌 값도 처리하기 위한 함수 정의
def safe_literal_eval(val):
    if isinstance(val, str):
        return ast.literal_eval(val)  # 문자열이면 딕셔너리로 변환
    return val  # 이미 딕셔너리면 그대로 반환

df[col_name] = df[col_name].apply(safe_literal_eval)

# 1. score 평균값 계산
df['score'] = df[col_name].apply(lambda x: x['score'])
score_mean = df['score'].mean()

# 2. reason이 '해당없음'가 아닌 행
df['reason'] = df[col_name].apply(lambda x: x['reason'])
non_x_reasons = df[df['reason'] != '해당없음']

# 3. score 상위 3개의 행
top3_scores = df.nsmallest(10, 'score')

# 결과 출력
print(f"Score 평균값: {score_mean}")

for i in range(len(top3_scores)):
    print(top3_scores.iloc[i].user_info)
    print(top3_scores.iloc[i].recipe_info)
    print(top3_scores.iloc[i].generation)
    print(top3_scores.iloc[i].score)
    print(top3_scores.iloc[i].reason)
    print()


Score 평균값: 0.9314285714285715
{'user_allergy_ingredients': ['새우', '게', '견과류'], 'user_dislike_ingredients': ['닭고기', '브로콜리', '오이'], 'user_spicy_level': '5단계', 'user_cooking_level': '고급', 'user_owned_ingredients': ['연어', '아스파라거스', '파스타면'], 'user_basic_seasoning': ['버터', '허브솔트', '파마산 치즈'], 'must_use_ingredients': ['아스파라거스']}
{'_id': '6761069a846f9e5eb976030a', 'title': '\ufeff편스토랑이찬원 멸치고추다짐장 레시피 만드는법', 'type_key': '반찬', 'method_key': '조림', 'servings': '4인분', 'cooking_time': '30분 이내', 'difficulty': '초급', 'ingredients': ['멸치(60마리)', '아삭이고추(2개)', '홍고추(2개)', '청양고추(16개)', '식용유(4큰술)', '다진 마늘(4큰술)', '국간장(4큰술)', '멸치액젓(3큰술)', '매실액(1큰술)', '참기름(2큰술)', '물(1/2컵)'], 'cooking_steps': ['멸치를 내장과 머리를 제거하고 다져 줍니다. 그리고 마른 팬에 5분 정도 볶아서 비린내와 수분을 날려 주세요.약 불에서 볶아 주세요.', '고추는 모두 잘게 다져 줍니다.', '다시마로 육수를 내어 줍니다.센 불에서 끓여 주세요.', '달궈 진 팬에 식용유를 두르고 다진 마늘을 넣어서 볶아 줍니다. 그리고 식용유에 마늘 향이 베이면 다진 고추를 넣고 볶아 주세요.중 약불에서 볶아 주세요', '고추가 어느 정도 볶아지면 멸치를 넣은 후 양념 재료와 다시마 육수를 적당히 넣어서 볶아 줍니다.중 약불에서 볶아 주세요.', ' 참기름을 넣고 한번 더 볶은 후 불을 꺼 줍니다.중 약

### 2. eval prompt로 test - conciseness 
- 2번만 진행

In [1]:
from langchain_teddynote import logging
# set_enable=False 로 지정하면 추적을 하지 않습니다.
logging.langsmith("랭체인 튜토리얼 프로젝트", set_enable=False)

import nest_asyncio
import asyncio
import pandas as pd
import sys
sys.path.append('../')  # 상위 디렉토리의 src 폴더를 경로에 추가
from src.recipe_change_origin import eval_recipe

# nest_asyncio로 이미 실행 중인 루프에서 중첩 실행 허용
nest_asyncio.apply()

# 동시 요청 제한과 재시도 설정
MAX_CONCURRENT_REQUESTS = 3
RETRY_LIMIT = 3

async def generate(df, prompt_name, col_name):
    df[col_name] = None  # 결과 저장 열 생성

    # Semaphore 생성
    semaphore = asyncio.Semaphore(MAX_CONCURRENT_REQUESTS)

    tasks = []  # 모든 작업을 저장할 리스트

    for i in range(len(df)):
        retry_count = 0
        while retry_count <= RETRY_LIMIT:
            try:
                # Semaphore로 동시 요청 제한
                async with semaphore:
                    print(f"Processing row {i} (Attempt {retry_count + 1})...")
                    result = await eval_recipe(df.iloc[i].recipe_info, df.iloc[i].user_info, df.iloc[i].generation, df.iloc[i].groundTruth, prompt_name)
                    df.at[i, col_name] = result  # 결과 저장
                    break  # 성공하면 반복문 종료
            except Exception as e:
                retry_count += 1
                if retry_count > RETRY_LIMIT:
                    print(f"Failed to process row {i} after {RETRY_LIMIT} retries.")
                    break  # 재시도 초과 시 반복문 종료
                print(f"Error processing row {i}: {e}. Retrying...")
                await asyncio.sleep(10 ** retry_count)  # 재시도 전 대기

        tasks.append(asyncio.sleep(0))  # 리스트에 임시 작업 추가 (추후 확장 가능)

    await asyncio.gather(*tasks)  # 모든 작업 실행

    # 결과 확인
    print(df.head())
    


LangSmith 추적을 하지 않습니다.


2024-12-23 08:15:48,389 - recipe_logger - INFO - LLM 초기화 완료: gpt-4o


In [2]:
# 데이터 로드
feature_num = 2  # 사용할 feature_num 설정
prompt_name = col_name = "conciseness"

file_path = f'data/groundTruth/final_groundTruth_f{feature_num}.csv'
df = pd.read_csv(file_path)

# 이벤트 루프 생성 및 실행
loop = asyncio.get_event_loop()
loop.run_until_complete(generate(df, prompt_name, col_name))

# 결과 저장
output_file = f'data/groundTruth/final_groundTruth_f{feature_num}.csv'
df.to_csv(output_file, index=False, encoding='utf-8-sig')
print(f"Processed data saved to {output_file}")

2024-12-23 08:15:48,824 - recipe_logger - INFO - json 출력 파서 초기화 완료.


Processing row 0 (Attempt 1)...


2024-12-23 08:15:49,428 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 08:15:49,432 - recipe_logger - INFO - LLM 레시피 생성 중...
2024-12-23 08:15:50,622 - recipe_logger - INFO - LLM 레시피 생성 완료
2024-12-23 08:15:50,636 - recipe_logger - INFO - json 출력 파서 초기화 완료.
2024-12-23 08:15:50,637 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 08:15:50,638 - recipe_logger - INFO - LLM 레시피 생성 중...


Processing row 1 (Attempt 1)...


2024-12-23 08:15:51,887 - recipe_logger - INFO - LLM 레시피 생성 완료
2024-12-23 08:15:51,897 - recipe_logger - INFO - json 출력 파서 초기화 완료.
2024-12-23 08:15:51,898 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 08:15:51,899 - recipe_logger - INFO - LLM 레시피 생성 중...


...

Processing row 33 (Attempt 1)...


2024-12-23 08:17:33,212 - recipe_logger - INFO - LLM 레시피 생성 완료
2024-12-23 08:17:33,228 - recipe_logger - INFO - json 출력 파서 초기화 완료.
2024-12-23 08:17:33,229 - recipe_logger - INFO - langfuse prompt template 생성 완료
2024-12-23 08:17:33,230 - recipe_logger - INFO - LLM 레시피 생성 중...


Processing row 34 (Attempt 1)...


2024-12-23 08:17:38,546 - recipe_logger - INFO - LLM 레시피 생성 완료


                                           user_info  \
0  {'user_allergy_ingredients': [], 'user_dislike...   
1  {'user_allergy_ingredients': [], 'user_dislike...   
2  {'user_allergy_ingredients': [], 'user_dislike...   
3  {'user_allergy_ingredients': [], 'user_dislike...   
4  {'user_allergy_ingredients': ['우유', '견과류'], 'u...   

                                         recipe_info  \
0  {'_id': '67610699846f9e5eb975e532', 'title': '...   
1  {'_id': '6761069a846f9e5eb9761021', 'title': '...   
2  {'_id': '6761069a846f9e5eb9760836', 'title': '...   
3  {'_id': '6761069a846f9e5eb9760639', 'title': '...   
4  {'_id': '6761069a846f9e5eb976093b', 'title': '...   

                                          generation  \
0  {'main_changes_from_original_recipe': ['연어 대신 ...   
1  {'main_changes_from_original_recipe': ['🍇 포도주스...   
2  {'main_changes_from_original_recipe': ['양파와 감자...   
3  {'main_changes_from_original_recipe': ['레몬을 굵은...   
4  {'main_changes_from_original_recipe': ['소고기

In [4]:
import pandas as pd
import ast

# 데이터 로드
feature_num = 2  # 사용할 feature_num 설정
prompt_name = col_name = "conciseness"

file_path = f'data/groundTruth/final_groundTruth_f{feature_num}.csv'
df = pd.read_csv(file_path)

# 문자열이 아닌 값도 처리하기 위한 함수 정의
def safe_literal_eval(val):
    if isinstance(val, str):
        return ast.literal_eval(val)  # 문자열이면 딕셔너리로 변환
    return val  # 이미 딕셔너리면 그대로 반환

df[col_name] = df[col_name].apply(safe_literal_eval)

# 1. score 평균값 계산
df['score'] = df[col_name].apply(lambda x: x['score'])
score_mean = df['score'].mean()

# 2. reason이 '해당없음'가 아닌 행
df['reason'] = df[col_name].apply(lambda x: x['reason'])
non_x_reasons = df[df['reason'] != '해당없음']

# 3. score 상위 3개의 행
top3_scores = df.nsmallest(10, 'score')

# 결과 출력
print(f"Score 평균값: {score_mean}")

for i in range(len(top3_scores)):
    print(top3_scores.iloc[i].user_info)
    print(top3_scores.iloc[i].recipe_info)
    print(top3_scores.iloc[i].generation)
    print(top3_scores.iloc[i].score)
    print(top3_scores.iloc[i].reason)
    print()


Score 평균값: 0.8257142857142858
{'user_allergy_ingredients': [], 'user_dislike_ingredients': [], 'user_spicy_level': '3단계', 'user_cooking_level': '중급', 'user_owned_ingredients': ['닭가슴살', '두부', '브로콜리'], 'user_basic_seasoning': ['소금', '후추', '올리브유'], 'must_use_ingredients': ['닭가슴살']}
{'_id': '6761069a846f9e5eb9761fe5', 'title': '저어서 만들어야 하는 수플레오믈렛 만들기', 'type_key': '퓨전', 'method_key': '굽기', 'servings': '2인분', 'cooking_time': '30분 이내', 'difficulty': '중급', 'ingredients': ['달걀(3개)', '생크림(4숟가락)', '설탕(4숟가락)', '버터(약간)', '메이플시럽(적당량)', '과일(적당량)'], 'cooking_steps': ['달걀은 흰자와 노른자를 분리한다.', '노른자를 살짝 풀어 준 후 생크림을 넣고 살짝 거품이 올라올 정도로 휘핑한다.', '흰자에 설탕을 넣어가며 휘핑하여 머랭을 만든다.', '휘핑 한 노른자에 머랭 1/2분량을 넣고 섞어 반죽을 만든다.', '달군 팬에 버터를 녹인 후 반죽을 올려 약 불로 5분간 익힌다.', '밑면과 가장자리가 노릇해지면 남겨둔 머랭을 반만 올린 후 반으로 접는다.', '뚜껑을 덮고 약 불에서 5분간 더 익힌다.', '접시에 수플레 오믈렛을 담고 메이플 시럽, 과일을 곁들여 완성한다'], 'tips': []}
{'main_changes_from_original_recipe': ['닭가슴살은 수플레의 부드러운 질감을 해칠 수 있어 제외했어요! 🥳 대신, 수플레 오믈렛을 더욱 부드럽고 가벼운 맛으로 유지했답니다. 🍳✨', '설탕과 메이플 시럽은 그대로 두어 달콤